# Setup

Install graphrag-toolkit-document-graph:

In [ ]:
!pip install graphrag-toolkit-document-graph

## Graph Store

### Amazon Neptune Database

If you are using Amazon Neptune Database as a graph store, install the lexical-graph dependency (provides `GraphStoreFactory`):

In [ ]:
!pip install graphrag-toolkit-document-graph[graphrag]

### Neo4j

If you are using Neo4j as a graph store:

In [ ]:
!pip install graphrag-toolkit-document-graph[neo4j]

## Hybrid Graph (document-graph + lexical-graph)

If you plan to use document-graph alongside lexical-graph for hybrid structured + unstructured graphs, install both:

In [ ]:
!pip install graphrag-toolkit-lexical-graph
!pip install graphrag-toolkit-document-graph[graphrag]

## Verify Installation

In [ ]:
from graphrag_toolkit.document_graph import Node, Edge
from graphrag_toolkit.document_graph.graph_build.cypher_builder import node_to_cypher, edge_to_cypher
from graphrag_toolkit.document_graph.query import DocumentGraphQueryEngine

print('graphrag-toolkit-document-graph ✅')

## Verify Neptune Connection

Replace the endpoint below with your Neptune cluster endpoint. You can find this in the Neptune console under your cluster's **Connectivity & security** tab.

In [ ]:
import os
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory

GRAPH_STORE = os.environ.get(
    'GRAPH_STORE',
    'neptune-db://<your-neptune-cluster-endpoint>:8182'
)

graph_store = GraphStoreFactory.for_graph_store(GRAPH_STORE)
gs = graph_store.__enter__()

result = gs.execute_query('MATCH (n) RETURN count(n) as cnt LIMIT 1')
print(f'Neptune connected ✅ — node count: {result[0]["cnt"]}')

## Write a Test Node

Write a single test node to verify end-to-end write and read.

In [ ]:
node = Node(
    id='setup-test-001',
    labels=['TestNode'],
    properties={'source': 'setup-notebook', 'status': 'ok'}
)

cypher, params = node_to_cypher(node, tenant_id='test')
print(f'Cypher: {cypher}')

gs.execute_query(cypher, params)
print('Write ✅')

# Read back
engine = DocumentGraphQueryEngine(gs, tenant_id='test')
result = engine.find_by_property('TestNode', 'source', 'setup-notebook')
print(f'Read  ✅ — {result}')

# Clean up context manager
graph_store.__exit__(None, None, None)